In [ ]:
# Run once: install pyyaml if not already installed
!pip install pyyaml

In [13]:
#Finding where am I
import os
os.environ['USE_LOCALSTACK'] = '1'
os.environ['LOCALSTACK_URL'] = 'http://localhost:4566'
# os.chdir('/config/workspace/python/pythonawsproject')
# print(os.getcwd())

# Now import (after chdir so Python can find the network package)
from network.state import load_state
print(load_state())

{'vpc_id': 'vpc-a56dacaa', 'vpc_name': 'dev-vpc', 'public-subnet-az1': 'subnet-8a7aaab9', 'public-subnet-az2': 'subnet-f81c0999', 'private-subnet-az1': 'subnet-cefc6cdc', 'private-subnet-az2': 'subnet-c0832271', 'igw_id': 'igw-2ef99f3d', 'public_rt_id': 'rtb-fd0bba37', 'public-subnet-az1_rt_assoc': 'rtbassoc-5dcf8aed', 'public-subnet-az2_rt_assoc': 'rtbassoc-86316be9', 'private_rt_id': 'rtb-faf6ec25', 'private-subnet-az1_rt_assoc': 'rtbassoc-1e88ae4d', 'private-subnet-az2_rt_assoc': 'rtbassoc-e81a84be', 'nat_eip_alloc_id': 'eipalloc-b4db550b6c1ff42fa', 'nat_id': 'nat-316bfee1ec39ae080', 'private_nat_route': True, 'sg_dev-web-sg': 'sg-2478c6e3cc81d3618', 'ec2_instance_id': 'i-ddb9399766782bb77'}


In [14]:
#What's the plan yo!
import yaml

with open('config/network.yaml', 'r') as f:
    config = yaml.safe_load(f) #//putting all the values from load state to config

print(config['vpc'])
print(config['security_groups'])

{'name': 'dev-vpc', 'cidr': '10.0.0.0/16', 'region': 'ap-south-1'}
[{'name': 'dev-web-sg', 'description': 'Allow SSH, HTTP, HTTPS inbound', 'ingress': [{'protocol': 'tcp', 'port': 22, 'cidr': '0.0.0.0/0', 'description': 'SSH access'}, {'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}, {'protocol': 'tcp', 'port': 443, 'cidr': '0.0.0.0/0', 'description': 'HTTPs'}], 'egress': [{'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}]}]


In [15]:
##Now it can work
from network.vpc import create_vpc

vpc_id = create_vpc(config)
print(f"Result: {vpc_id}")

AttributeError: 'EC2' object has no attribute 'describe_vpc'

In [4]:
from network.subnets import create_subnet
##We are utting full config python file here for execution
result = create_subnet(config)
print(result)

[{'name': 'public-subnet-az1', 'cidr': '10.0.1.0/24', 'az': 'ap-south-1a'}, {'name': 'public-subnet-az2', 'cidr': '10.0.2.0/24', 'az': 'ap-south-1b'}, {'name': 'private-subnet-az1', 'cidr': '10.0.3.0/24', 'az': 'ap-south-1a'}, {'name': 'private-subnet-az2', 'cidr': '10.0.4.0/24', 'az': 'ap-south-1b'}]
Subnet already Exsists by name: public-subnet-az1
Subnet already Exsists by name: public-subnet-az2
Subnet already Exsists by name: private-subnet-az1
Subnet already Exsists by name: private-subnet-az2


NoCredentialsError: Unable to locate credentials

In [5]:
from network.internet_gateway  import creating_igw
from network.state import load_state, save_state
from network.aws_client import get_client

igw_id = creating_igw(config)
print(f"Result: {igw_id}")
# ec2 = get_client('ec2')
# response = ec2.describe_internet_gateways(
#     Filters=[{'Name': 'attachment.vpc-id', 'Values': [vpc_id]}]
# )

# print(vpc_id)

# response = ec2.describe_internet_gateways()
# print(response['InternetGateways'])

# igw_id = response['InternetGateways'][0]['InternetGatewayId']
# print(f"Found exsisting IGW: {igw_id}")

# ec2.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
# print(f"Attached {igw_id} to {vpc_id}")

# ##Saving and loading of state file
# state = load_state()
# state['igw_id'] = igw_id
# save_state(state)
# print("Saved State")

# igw_id = creating_igw(config)
# print(f"Result: {igw_id}")

IGW already exsists: igw-2ef99f3d
Result: igw-2ef99f3d


In [6]:
from network.route_tables import create_route_tables

result = create_route_tables(config)

Public route table already exsists: rtb-fd0bba37
 Already associated: public-subnet-az1
 Already associated: public-subnet-az2
Private route table already exists: rtb-faf6ec25
  Already associated: private-subnet-az1
  Already associated: private-subnet-az2
Route tables done.


In [7]:
from network.nat_gateway import create_nat_gateway

result = create_nat_gateway(config)

Elastic IP already exists: eipalloc-b4db550b6c1ff42fa
NAT Gateway already exists: nat-316bfee1ec39ae080
Private route to NAT already exists
NAT Gateway done.


In [8]:
from network.state import load_state, save_state

state = load_state()
del state['nat_id']
del state['nat_eip_alloc_id']

In [9]:
import os
os.chdir('/workspaces/floci-boto-3-python')
print(os.listdir('network/'))

['ec2_instance.py', 'key_pair.py', 'state.py', 'main.py', '__pycache__', 'nat_gateway.py', 'security_groups.py', 'internet_gateway.py', 'subnets.py', '__init__.py', 'route_tables.py', 'aws_client.py', 'planner.py', 'vpc.py']


In [10]:
from network.security_groups import create_security_groups
create_security_groups(config)

Security Group already exists: sg-2478c6e3cc81d3618 (dev-web-sg)
Security Groups done.


{'vpc_id': 'vpc-a56dacaa',
 'vpc_name': 'dev-vpc',
 'public-subnet-az1': 'subnet-8a7aaab9',
 'public-subnet-az2': 'subnet-f81c0999',
 'private-subnet-az1': 'subnet-cefc6cdc',
 'private-subnet-az2': 'subnet-c0832271',
 'igw_id': 'igw-2ef99f3d',
 'public_rt_id': 'rtb-fd0bba37',
 'public-subnet-az1_rt_assoc': 'rtbassoc-5dcf8aed',
 'public-subnet-az2_rt_assoc': 'rtbassoc-86316be9',
 'private_rt_id': 'rtb-faf6ec25',
 'private-subnet-az1_rt_assoc': 'rtbassoc-1e88ae4d',
 'private-subnet-az2_rt_assoc': 'rtbassoc-e81a84be',
 'nat_eip_alloc_id': 'eipalloc-b4db550b6c1ff42fa',
 'nat_id': 'nat-316bfee1ec39ae080',
 'private_nat_route': True,
 'sg_dev-web-sg': 'sg-2478c6e3cc81d3618',
 'ec2_instance_id': 'i-ddb9399766782bb77'}

In [12]:
from network.key_pair import create_key_pair
create_key_pair(config)

NoCredentialsError: Unable to locate credentials

In [45]:

from network.ec2_instance import create_ec2_instance
instance_id = create_ec2_instance(config)
print(f"Result: {instance_id}")

Launched EC2 instance: i-ddb9399766782bb77
Waiting for instance to reach 'running' state...
Instance is now running.
Save EC@ instance to state: i-ddb9399766782bb77
Result: i-ddb9399766782bb77


In [46]:
from network.planner import run_planner
plan = run_planner()

🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-a56dacaa...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-8a7aaab9...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-f81c0999...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-cefc6cdc...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-c0832271...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-2ef99f3d...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-316bfee1ec39ae08...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-fd0bba37...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-faf6ec25...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-2478c6e3cc81d3618...)
  0 to create, 0 to delete, 10 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.


In [47]:
from network.main import command_plan
command_plan(config)


 Scanning the  battlefiend that is config file
🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-a56dacaa...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-8a7aaab9...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-f81c0999...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-cefc6cdc...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-c0832271...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-2ef99f3d...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-316bfee1ec39ae08...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-fd0bba37...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-faf6ec25...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-2478c6e3cc81d3618...)
  0 to create, 0 to delete, 10 unchanged

  ✅ Infrastructure matches desired state. Nothing to do.


[{'type': 'vpc', 'name': 'dev-vpc', 'action': 'NO-OP', 'id': 'vpc-a56dacaa'},
 {'type': 'subnet',
  'name': 'public-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-8a7aaab9'},
 {'type': 'subnet',
  'name': 'public-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-f81c0999'},
 {'type': 'subnet',
  'name': 'private-subnet-az1',
  'action': 'NO-OP',
  'id': 'subnet-cefc6cdc'},
 {'type': 'subnet',
  'name': 'private-subnet-az2',
  'action': 'NO-OP',
  'id': 'subnet-c0832271'},
 {'type': 'internet_gateway',
  'name': 'dev-igw',
  'action': 'NO-OP',
  'id': 'igw-2ef99f3d'},
 {'type': 'nat_gateway',
  'name': 'dev-nat',
  'action': 'NO-OP',
  'id': 'nat-316bfee1ec39ae080'},
 {'type': 'route_table',
  'name': 'public-route-table',
  'action': 'NO-OP',
  'id': 'rtb-fd0bba37'},
 {'type': 'route_table',
  'name': 'private-route-table',
  'action': 'NO-OP',
  'id': 'rtb-faf6ec25'},
 {'type': 'security_group',
  'name': 'dev-web-sg',
  'action': 'NO-OP',
  'id': 'sg-2478c6e3cc81d3618'}]

In [34]:
from network.main import command_destroy
command_destroy(config)


 RETREAT ORDER - Demolishing infrastructure ...

----Terminate EC2 Instance------
  Waiting for i-77759240227fdfda9 to terminate...
  EC2 instance terminated.
===== Destroying NAT Gateway =====
  Waiting for NAT nat-37d6e6198e8052536 to die...
  NAT Gateway deleted.
===== Releasing Elastic IP =====
  EIP released. Thanks for choosing the service.
===== Disassociating Route Tables =====
  Unfollowing: public-subnet-az1_rt_assoc
  Unfollowing: public-subnet-az2_rt_assoc
  Unfollowing: private-subnet-az1_rt_assoc
  Unfollowing: private-subnet-az2_rt_assoc
===== Deleting Route Tables =====
  Shredding: rtb-be39eaa0
  Shredding: rtb-2fb22733
===== Detaching + Deleting IGW =====
  IGW igw-7719e70e unhooked from VPC.
  IGW igw-7719e70e deleted.
  IGW gone. No more front door.
 Deleteing Security Group: sg-82153192c4007fdab (sg_dev-web-sg)
===== Deleting Subnets =====
  Demolishing: subnet-5f133b96 (public-subnet-az1)
  Demolishing: subnet-fa1d8827 (public-subnet-az2)
  Demolishing: subnet-e8

In [18]:
command_plan(config)


 Scanning the  battlefiend that is config file
🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ⚡ vpc                  "dev-vpc" → NEEDS CREATION
  ⚡ subnet               "public-subnet-az1" → NEEDS CREATION
  ⚡ subnet               "public-subnet-az2" → NEEDS CREATION
  ⚡ subnet               "private-subnet-az1" → NEEDS CREATION
  ⚡ subnet               "private-subnet-az2" → NEEDS CREATION
  ⚡ internet_gateway     "dev-igw" → NEEDS CREATION
  ⚡ nat_gateway          "dev-nat" → NEEDS CREATION
  ⚡ route_table          "public-route-table" → NEEDS CREATION
  ⚡ route_table          "private-route-table" → NEEDS CREATION
  ⚡ security_group       "dev-web-sg" → NEEDS CREATION
  10 to create, 0 to delete, 0 unchanged

  ⚡ Run 'python main.py apply' to execute this plan.


[{'type': 'vpc', 'name': 'dev-vpc', 'action': 'CREATE', 'id': None},
 {'type': 'subnet',
  'name': 'public-subnet-az1',
  'action': 'CREATE',
  'id': None},
 {'type': 'subnet',
  'name': 'public-subnet-az2',
  'action': 'CREATE',
  'id': None},
 {'type': 'subnet',
  'name': 'private-subnet-az1',
  'action': 'CREATE',
  'id': None},
 {'type': 'subnet',
  'name': 'private-subnet-az2',
  'action': 'CREATE',
  'id': None},
 {'type': 'internet_gateway',
  'name': 'dev-igw',
  'action': 'CREATE',
  'id': None},
 {'type': 'nat_gateway', 'name': 'dev-nat', 'action': 'CREATE', 'id': None},
 {'type': 'route_table',
  'name': 'public-route-table',
  'action': 'CREATE',
  'id': None},
 {'type': 'route_table',
  'name': 'private-route-table',
  'action': 'CREATE',
  'id': None},
 {'type': 'security_group',
  'name': 'dev-web-sg',
  'action': 'CREATE',
  'id': None}]